## Trade Timestamp Occurrence Pattern

This notebook checks whether trade occurrences follow a timestamp pattern in `trades_round_0_day_-1.csv`.

It focuses on:
- timestamp granularity
- inter-trade gap distribution
- repeated timestamps (multiple trades at same time)
- clustering over time

In [9]:
import pandas as pd

# Load trade data
trades = pd.read_csv("trades_round_0_day_-1.csv", sep=";")
trades["timestamp"] = trades["timestamp"].astype(int)

# Basic timestamp stats
ts = trades["timestamp"].sort_values().reset_index(drop=True)
print(f"rows: {len(ts)}")
print(f"unique timestamps: {ts.nunique()}")
print(f"time range: {ts.min()} -> {ts.max()}")

# 1) Granularity / quantization check
mod_counts = ts.mod(100).value_counts().sort_index()
print("\nmod 100 counts:")
print(mod_counts)

# 2) Inter-trade gaps (sorted by time; includes 0 when multiple trades share same timestamp)
gaps = ts.diff().dropna().astype(int)
print("\nTop 10 most common gaps:")
print(gaps.value_counts().head(10))
print(f"median gap: {gaps.median()}")

# 3) Duplicate timestamps (multiple trades at same timestamp)
dup_ts = ts.value_counts()
dup_ts = dup_ts[dup_ts > 1].sort_values(ascending=False)
print(f"\nnumber of timestamps with >1 trade: {len(dup_ts)}")
print("Top duplicate timestamps:")
print(dup_ts.head(10))

# 4) Clustering by coarse time bucket
bucket = 10_000
trades["time_bucket"] = (trades["timestamp"] // bucket) * bucket
bucket_counts = trades["time_bucket"].value_counts().sort_index()
print(f"\nTrades per {bucket}-time-unit bucket:")
print(bucket_counts)

# 5) Per-symbol cadence
print("\nPer-symbol trade counts:")
print(trades["symbol"].value_counts())

print("\nPer-symbol median inter-trade gap:")
for symbol, group in trades.sort_values("timestamp").groupby("symbol"):
    s = group["timestamp"].sort_values().reset_index(drop=True)
    g = s.diff().dropna()
    if len(g) > 0:
        print(f"{symbol}: median gap = {g.median():.1f}, mean gap = {g.mean():.1f}, trades = {len(s)}")

rows: 631
unique timestamps: 626
time range: 3200 -> 995600

mod 100 counts:
timestamp
0    631
Name: count, dtype: int64

Top 10 most common gaps:
timestamp
100     46
300     38
400     33
500     33
200     31
600     28
900     25
1100    25
800     23
700     22
Name: count, dtype: int64
median gap: 1100.0

number of timestamps with >1 trade: 5
Top duplicate timestamps:
timestamp
132100    2
212200    2
575900    2
751800    2
885900    2
Name: count, dtype: int64

Trades per 10000-time-unit bucket:
time_bucket
0         6
10000     2
20000     4
30000     6
40000     2
         ..
950000    6
960000    4
970000    5
980000    5
990000    2
Name: count, Length: 100, dtype: int64

Per-symbol trade counts:
symbol
TOMATOES    423
EMERALDS    208
Name: count, dtype: int64

Per-symbol median inter-trade gap:
EMERALDS: median gap = 3300.0, mean gap = 4745.9, trades = 208
TOMATOES: median gap = 1600.0, mean gap = 2351.2, trades = 423


### How To Read The Output

If the data follows the same behavior seen in this file, you should observe:
- timestamps quantized to a fixed step (most likely multiples of 100)
- irregular inter-trade gaps (no strict periodic schedule)
- burstiness, with some timestamps having multiple trades at once (gap = 0)
- activity clustering in certain time windows rather than uniform spacing

This indicates event-driven trade arrivals with a discrete simulation clock, not a constant-frequency process.

### Uniform Randomness Check

This section tests whether trade timestamps are consistent with a uniform random distribution over the observed time range.

Methods used:
- Chi-square goodness-of-fit on equal-width time bins
- Monte Carlo p-value for the same statistic (robust, no extra dependency needed)
- Dispersion index (variance/mean of bin counts):
  - near 1 suggests Poisson-like randomness
  - greater than 1 suggests clustering (bursty arrivals)

In [10]:
import numpy as np

# Use timestamps from the existing dataframe created above
x = trades["timestamp"].to_numpy(dtype=float)
n = x.size
xmin, xmax = x.min(), x.max()

# Equal-width bins over observed range (choose enough counts per bin for stability)
num_bins = 20
edges = np.linspace(xmin, xmax + 1e-9, num_bins + 1)
obs, _ = np.histogram(x, bins=edges)
expected = n / num_bins

# Chi-square statistic
chi2_stat = np.sum((obs - expected) ** 2 / expected)

# Monte Carlo p-value under uniform timestamps
rng = np.random.default_rng(42)
num_sim = 5000
sim_stats = np.empty(num_sim)
for i in range(num_sim):
    sim = rng.uniform(xmin, xmax, size=n)
    sim_obs, _ = np.histogram(sim, bins=edges)
    sim_stats[i] = np.sum((sim_obs - expected) ** 2 / expected)

p_mc = (np.sum(sim_stats >= chi2_stat) + 1) / (num_sim + 1)

# Dispersion index: var/mean of bin counts (1 ~ random Poisson-like, >1 clustered)
dispersion_index = obs.var(ddof=1) / obs.mean()

print("Uniformity test on timestamp occurrences")
print(f"n trades: {n}")
print(f"time range: [{xmin:.0f}, {xmax:.0f}]")
print(f"bins: {num_bins}, expected per bin: {expected:.2f}")
print(f"chi2 statistic: {chi2_stat:.3f}")
print(f"Monte Carlo p-value: {p_mc:.4f}")
print(f"dispersion index (var/mean): {dispersion_index:.3f}")
print("observed bin counts:", obs.tolist())

alpha = 0.05
if p_mc < alpha:
    print("Conclusion: Reject uniform-random timestamp occurrence pattern at 5% significance.")
else:
    print("Conclusion: Cannot reject uniform-random timestamp occurrence pattern at 5% significance.")

if dispersion_index > 1.2:
    print("Additional signal: counts are over-dispersed (clustered/bursty).")
elif dispersion_index < 0.8:
    print("Additional signal: counts are under-dispersed (too regular).")
else:
    print("Additional signal: dispersion is close to random baseline.")

Uniformity test on timestamp occurrences
n trades: 631
time range: [3200, 995600]
bins: 20, expected per bin: 31.55
chi2 statistic: 13.786
Monte Carlo p-value: 0.7982
dispersion index (var/mean): 0.726
observed bin counts: [22, 39, 32, 30, 32, 28, 30, 32, 41, 31, 27, 32, 36, 28, 32, 38, 31, 32, 35, 23]
Conclusion: Cannot reject uniform-random timestamp occurrence pattern at 5% significance.
Additional signal: counts are under-dispersed (too regular).


In [11]:
import numpy as np
import pandas as pd

# Work on occurrence process (at least one trade at a timestamp)
tick_size = 100
u = np.sort(trades["timestamp"].astype(int).unique())

# Confirm discretization to simulation ticks
all_on_tick = np.all(u % tick_size == 0)

# Build full tick grid and Bernoulli event indicator
all_ticks = np.arange(u.min(), u.max() + tick_size, tick_size)
event_indicator = pd.Series(0, index=all_ticks, dtype=int)
event_indicator.loc[u] = 1

# Waiting times between event ticks (in ticks)
wait_ticks = np.diff(u) // tick_size

# Basic diagnostics
p_hat = event_indicator.mean()  # Bernoulli per tick MLE
mean_wait = wait_ticks.mean()
std_wait = wait_ticks.std(ddof=1)
cv_wait = std_wait / mean_wait

# If events were fixed-delay, CV would be near 0. If Bernoulli per tick, CV ~= sqrt(1-p).
cv_geometric_theory = np.sqrt(1 - p_hat)

# Log-likelihood under geometric waiting model (support k>=1)
p_geo = 1.0 / mean_wait
ll_geo = np.sum(np.log(p_geo) + (wait_ticks - 1) * np.log(1 - p_geo))

# Log-likelihood under exponential waiting model on continuous time
wait_time = wait_ticks * tick_size
lam = 1.0 / wait_time.mean()
ll_exp = len(wait_time) * np.log(lam) - lam * wait_time.sum()

# Count process check in fixed windows (Poisson would have Var/Mean ~ 1)
window_ticks = 10  # 10 ticks = 1000 time units
window_size = window_ticks * tick_size
window_id = ((trades["timestamp"].astype(int) - u.min()) // window_size).astype(int)
counts_per_window = window_id.value_counts().sort_index().to_numpy()
vmr = counts_per_window.var(ddof=1) / counts_per_window.mean()  # variance-to-mean ratio

# Serial dependence check for Bernoulli i.i.d assumption
x = event_indicator.to_numpy(dtype=float)
if len(x) > 1:
    lag1_corr = np.corrcoef(x[:-1], x[1:])[0, 1]
else:
    lag1_corr = np.nan

print("Arrival Process Diagnostics")
print(f"all timestamps on {tick_size}-unit grid: {all_on_tick}")
print(f"active ticks: {event_indicator.sum()} / total ticks: {len(event_indicator)}")
print(f"p_hat (trade-occurrence probability per tick): {p_hat:.4f}")
print()
print("Waiting-time diagnostics (between active ticks):")
print(f"mean wait (ticks): {mean_wait:.3f}")
print(f"std wait (ticks): {std_wait:.3f}")
print(f"CV wait: {cv_wait:.3f}")
print(f"CV expected under Bernoulli/geometric: {cv_geometric_theory:.3f}")
print("(fixed-delay process would have CV near 0)")
print()
print("Model fit signals:")
print(f"geometric waiting log-likelihood: {ll_geo:.2f}")
print(f"exponential waiting log-likelihood: {ll_exp:.2f}")
print(f"lag-1 autocorrelation of occurrence indicator: {lag1_corr:.3f}")
print(f"window count variance/mean ratio (Poisson baseline = 1): {vmr:.3f}")

print("\nInterpretation guide:")
print("- If CV is far above geometric expectation or lag-1 correlation is positive, arrivals are clustered.")
print("- If variance/mean ratio >> 1, counts are over-dispersed vs Poisson.")
print("- If fixed-delay were true, waits would be tightly concentrated with CV near 0.")

Arrival Process Diagnostics
all timestamps on 100-unit grid: True
active ticks: 626 / total ticks: 9925
p_hat (trade-occurrence probability per tick): 0.0631

Waiting-time diagnostics (between active ticks):
mean wait (ticks): 15.878
std wait (ticks): 15.082
CV wait: 0.950
CV expected under Bernoulli/geometric: 0.968
(fixed-delay process would have CV near 0)

Model fit signals:
geometric waiting log-likelihood: -2332.99
exponential waiting log-likelihood: -5231.33
lag-1 autocorrelation of occurrence indicator: 0.011
window count variance/mean ratio (Poisson baseline = 1): 0.263

Interpretation guide:
- If CV is far above geometric expectation or lag-1 correlation is positive, arrivals are clustered.
- If variance/mean ratio >> 1, counts are over-dispersed vs Poisson.
- If fixed-delay were true, waits would be tightly concentrated with CV near 0.


In [12]:
import numpy as np
import math

# Model comparison on waiting times between active timestamps.
# wait_ticks should already exist from Cell 6. Recompute defensively if needed.
try:
    wait_ticks
except NameError:
    u = np.sort(trades["timestamp"].astype(int).unique())
    wait_ticks = np.diff(u) // 100

k = np.asarray(wait_ticks, dtype=int)
n = len(k)

if n < 5:
    print("Not enough waiting-time observations for model comparison.")
else:
    # ---------- Model 1: Geometric on k>=1 ----------
    # P(K=k)=p(1-p)^(k-1), E[K]=1/p
    p_hat = 1.0 / k.mean()
    ll_geo = np.sum(np.log(p_hat) + (k - 1) * np.log(1 - p_hat))
    aic_geo = 2 * 1 - 2 * ll_geo
    bic_geo = np.log(n) * 1 - 2 * ll_geo

    # ---------- Model 2: Exponential on continuous wait time ----------
    # Continuous-time analogue; included for reference.
    tick_size = 100.0
    t = k * tick_size
    lam_hat = 1.0 / t.mean()
    ll_exp = n * np.log(lam_hat) - lam_hat * t.sum()
    aic_exp = 2 * 1 - 2 * ll_exp
    bic_exp = np.log(n) * 1 - 2 * ll_exp

    # ---------- Model 3: Fixed-delay baseline ----------
    # "Wait x ticks after last trade": if strictly deterministic, only one value can occur.
    # We report mismatch rate and SSE to show how far data is from this mechanism.
    d_hat = int(round(k.mean()))
    mismatch_rate = np.mean(k != d_hat)
    sse_fixed = np.sum((k - d_hat) ** 2)

    # ---------- Model 4: Negative Binomial on failures before success ----------
    # Let Y = K-1, Y~NB(r,p) with Var(Y) > Mean(Y) when over-dispersed.
    # Method-of-moments estimates; if variance <= mean, NB collapses toward geometric/Poisson edge.
    y = k - 1
    m = y.mean()
    v = y.var(ddof=1)

    nb_valid = v > m + 1e-12 and m > 0
    if nb_valid:
        r_hat = m * m / (v - m)
        p_nb = r_hat / (r_hat + m)

        # log pmf using lgamma for numerical stability
        ll_nb = 0.0
        for yi in y:
            ll_nb += (
                math.lgamma(yi + r_hat)
                - math.lgamma(r_hat)
                - math.lgamma(yi + 1)
                + r_hat * math.log(p_nb)
                + yi * math.log(1 - p_nb)
            )

        # 2 parameters: r, p
        aic_nb = 2 * 2 - 2 * ll_nb
        bic_nb = np.log(n) * 2 - 2 * ll_nb
    else:
        r_hat = np.nan
        p_nb = np.nan
        ll_nb = np.nan
        aic_nb = np.nan
        bic_nb = np.nan

    # ---------- Ranking ----------
    rows = [
        ("Geometric (discrete Bernoulli-per-tick)", ll_geo, aic_geo, bic_geo),
        ("Exponential (continuous-time reference)", ll_exp, aic_exp, bic_exp),
    ]
    if nb_valid:
        rows.append(("Negative Binomial (overdispersed waits)", ll_nb, aic_nb, bic_nb))

    rows_sorted = sorted(rows, key=lambda x: x[2])

    print("Model Comparison for Waiting Times")
    print(f"observations: {n}")
    print()
    print("AIC/BIC (lower is better):")
    for name, ll, aic, bic in rows_sorted:
        print(f"- {name}: ll={ll:.2f}, AIC={aic:.2f}, BIC={bic:.2f}")

    print()
    print("Fixed-delay baseline (wait exactly x ticks):")
    print(f"- best fixed delay d: {d_hat} ticks")
    print(f"- mismatch rate: {mismatch_rate:.3f}")
    print(f"- SSE around fixed delay: {sse_fixed:.1f}")

    print()
    print("Extra diagnostics:")
    print(f"- geometric p_hat per tick: {p_hat:.4f}")
    print(f"- mean wait: {k.mean():.3f} ticks")
    print(f"- std wait: {k.std(ddof=1):.3f} ticks")
    if nb_valid:
        print(f"- NB parameters (Y=K-1): r={r_hat:.3f}, p={p_nb:.3f}")
    else:
        print("- NB not supported by moments (no overdispersion vs geometric edge).")

    print()
    winner = rows_sorted[0][0]
    print(f"Most supported stochastic model by AIC: {winner}")
    if mismatch_rate > 0.10:
        print("Deterministic fixed-delay mechanism is unlikely.")
    else:
        print("Fixed-delay is not strongly contradicted by mismatch rate alone.")

Model Comparison for Waiting Times
observations: 625

AIC/BIC (lower is better):
- Geometric (discrete Bernoulli-per-tick): ll=-2332.99, AIC=4667.98, BIC=4672.42
- Negative Binomial (overdispersed waits): ll=-2333.63, AIC=4671.27, BIC=4680.14
- Exponential (continuous-time reference): ll=-5231.33, AIC=10464.66, BIC=10469.10

Fixed-delay baseline (wait exactly x ticks):
- best fixed delay d: 16 ticks
- mismatch rate: 0.965
- SSE around fixed delay: 141946.0

Extra diagnostics:
- geometric p_hat per tick: 0.0630
- mean wait: 15.878 ticks
- std wait: 15.082 ticks
- NB parameters (Y=K-1): r=1.041, p=0.065

Most supported stochastic model by AIC: Geometric (discrete Bernoulli-per-tick)
Deterministic fixed-delay mechanism is unlikely.


In [13]:
import numpy as np
import pandas as pd
from pathlib import Path
import re


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = (z / denom) * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))
    return center - half, center + half


def extract_day_label(file_name):
    m = re.search(r"day_(-?\d+)", file_name)
    return int(m.group(1)) if m else np.nan


def get_trade_files(glob_pattern="trades_round_0_day_*.csv"):
    return sorted([str(p) for p in Path(".").glob(glob_pattern)])


def estimate_symbol_p(df, symbol, tick_size=100):
    ts_all = df["timestamp"].astype(int)
    day_min, day_max = int(ts_all.min()), int(ts_all.max())
    total_ticks = len(np.arange(day_min, day_max + tick_size, tick_size))

    sdf = df[df["symbol"] == symbol]
    active_ticks = int(sdf["timestamp"].astype(int).nunique()) if not sdf.empty else 0

    p_hat = active_ticks / total_ticks if total_ticks > 0 else np.nan
    ci_low, ci_high = wilson_ci(active_ticks, total_ticks)

    return {
        "active_ticks": active_ticks,
        "total_ticks": total_ticks,
        "p_hat": p_hat,
        "ci_low": ci_low,
        "ci_high": ci_high,
    }


# Expandable configuration
symbols = ["TOMATOES", "EMERALDS"]
trade_files = get_trade_files("trades_round_0_day_*.csv")

# If you want specific files only, uncomment and edit:
# trade_files = ["trades_round_0_day_-1.csv", "trades_round_0_day_-2.csv"]

rows = []
for f in trade_files:
    df = pd.read_csv(f, sep=";")
    df["timestamp"] = df["timestamp"].astype(int)
    day_label = extract_day_label(Path(f).name)

    for sym in symbols:
        est = estimate_symbol_p(df, sym, tick_size=100)
        rows.append(
            {
                "file": Path(f).name,
                "day": day_label,
                "symbol": sym,
                **est,
            }
        )

result = pd.DataFrame(rows).sort_values(["day", "file", "symbol"]).reset_index(drop=True)

print("Per-day symbol occurrence probability")
print("Definition: p = P(at least one trade for symbol on a 100-time-unit tick)")
print()

# Primary output: explicit breakdown by file
for file_name, g in result.groupby("file", sort=False):
    day_val = g["day"].iloc[0]
    day_txt = f"day {int(day_val)}" if pd.notna(day_val) else "day unknown"
    print(f"File: {file_name} ({day_txt})")
    cols = ["symbol", "active_ticks", "total_ticks", "p_hat", "ci_low", "ci_high"]
    print(g[cols].to_string(index=False))

    if set(symbols).issubset(set(g["symbol"])):
        t = g[g["symbol"] == "TOMATOES"].iloc[0]
        e = g[g["symbol"] == "EMERALDS"].iloc[0]
        diff = t["p_hat"] - e["p_hat"]
        ratio = t["p_hat"] / e["p_hat"] if e["p_hat"] > 0 else np.nan
        print(f"Difference TOMATOES-EMERALDS: {diff:+.4f}")
        print(f"Ratio TOMATOES/EMERALDS: {ratio:.3f}")
    print("-" * 72)

print("\nAll rows (compact):")
print(result[["file", "day", "symbol", "active_ticks", "total_ticks", "p_hat", "ci_low", "ci_high"]])

print("\nPivot (p_hat by day):")
pivot = result.pivot_table(index="day", columns="symbol", values="p_hat", aggfunc="first").sort_index()
print(pivot)

print("\nPooled across selected files (by symbol):")
pool = result.groupby("symbol", as_index=False)[["active_ticks", "total_ticks"]].sum()
pool["p_hat"] = pool["active_ticks"] / pool["total_ticks"]
ci_vals = pool.apply(lambda r: wilson_ci(int(r["active_ticks"]), int(r["total_ticks"])), axis=1)
pool["ci_low"] = [c[0] for c in ci_vals]
pool["ci_high"] = [c[1] for c in ci_vals]
print(pool[["symbol", "active_ticks", "total_ticks", "p_hat", "ci_low", "ci_high"]])

Per-day symbol occurrence probability
Definition: p = P(at least one trade for symbol on a 100-time-unit tick)

File: trades_round_0_day_-2.csv (day -2)
  symbol  active_ticks  total_ticks    p_hat   ci_low  ci_high
EMERALDS           191         9909 0.019275 0.016749 0.022175
TOMATOES           396         9909 0.039964 0.036282 0.044002
Difference TOMATOES-EMERALDS: +0.0207
Ratio TOMATOES/EMERALDS: 2.073
------------------------------------------------------------------------
File: trades_round_0_day_-1.csv (day -1)
  symbol  active_ticks  total_ticks    p_hat   ci_low  ci_high
EMERALDS           208         9925 0.020957 0.018319 0.023966
TOMATOES           423         9925 0.042620 0.038819 0.046774
Difference TOMATOES-EMERALDS: +0.0217
Ratio TOMATOES/EMERALDS: 2.034
------------------------------------------------------------------------

All rows (compact):
                        file  day    symbol  active_ticks  total_ticks  \
0  trades_round_0_day_-2.csv   -2  EMERALDS      

In [14]:
import numpy as np
import pandas as pd

# Compare symbol behavior on the currently loaded trades dataframe.
# Assumes `trades` is already loaded from the earlier cells.

def symbol_breakdown(df, symbol, tick_size=100, global_min=None, global_max=None):
    s = df[df["symbol"] == symbol].copy()
    s["timestamp"] = s["timestamp"].astype(int)

    if s.empty:
        return {
            "symbol": symbol,
            "trades": 0,
        }

    ts_unique = np.sort(s["timestamp"].unique())
    waits = np.diff(ts_unique) // tick_size

    if global_min is None:
        global_min = int(df["timestamp"].min())
    if global_max is None:
        global_max = int(df["timestamp"].max())

    total_ticks_global = len(np.arange(global_min, global_max + tick_size, tick_size))
    p_tick_global = len(ts_unique) / total_ticks_global

    qty = s["quantity"].astype(float)
    px = s["price"].astype(float)

    dup = s.groupby("timestamp").size()
    multi_ts = int((dup > 1).sum())

    return {
        "symbol": symbol,
        "trades": int(len(s)),
        "active_timestamps": int(len(ts_unique)),
        "p_tick_global": float(p_tick_global),
        "mean_wait_ticks": float(waits.mean()) if len(waits) else np.nan,
        "std_wait_ticks": float(waits.std(ddof=1)) if len(waits) > 1 else np.nan,
        "median_wait_ticks": float(np.median(waits)) if len(waits) else np.nan,
        "total_quantity": float(qty.sum()),
        "mean_quantity": float(qty.mean()),
        "median_quantity": float(qty.median()),
        "price_min": float(px.min()),
        "price_median": float(px.median()),
        "price_mean": float(px.mean()),
        "price_max": float(px.max()),
        "price_std": float(px.std(ddof=1)) if len(px) > 1 else np.nan,
        "timestamps_with_multiple_trades": multi_ts,
    }

symbols = ["TOMATOES", "EMERALDS"]

global_min = int(trades["timestamp"].astype(int).min())
global_max = int(trades["timestamp"].astype(int).max())
rows = [symbol_breakdown(trades, sym, tick_size=100, global_min=global_min, global_max=global_max) for sym in symbols]
summary = pd.DataFrame(rows).set_index("symbol")

print("TOMATOES vs EMERALDS breakdown (current file loaded in `trades`)")
print(f"global timestamp range: {global_min} to {global_max}")
print("\nCore stats:")
print(summary[[
    "trades",
    "active_timestamps",
    "p_tick_global",
    "mean_wait_ticks",
    "median_wait_ticks",
    "std_wait_ticks",
    "total_quantity",
    "mean_quantity",
    "price_mean",
    "price_std",
]])

print("\nAdditional timing signal:")
for sym in symbols:
    r = summary.loc[sym]
    cv = r["std_wait_ticks"] / r["mean_wait_ticks"] if pd.notna(r["std_wait_ticks"]) and r["mean_wait_ticks"] > 0 else np.nan
    print(
        f"- {sym}: p_tick={r['p_tick_global']:.4f}, mean wait={r['mean_wait_ticks']:.2f}, "
        f"CV(wait)={cv:.3f}, multi-trade timestamps={int(r['timestamps_with_multiple_trades'])}"
    )

print("\nInterpretation hints:")
print("- Higher p_tick_global means more frequent occurrence on the global clock.")
print("- Lower mean_wait_ticks means trades happen more often for that symbol.")
print("- Higher CV(wait) suggests more irregular spacing of occurrences.")
print("- price_std compares how noisy each symbol's trade prices are.")

TOMATOES vs EMERALDS breakdown (current file loaded in `trades`)
global timestamp range: 3200 to 995600

Core stats:
          trades  active_timestamps  p_tick_global  mean_wait_ticks  \
symbol                                                                
TOMATOES     423                423       0.042620        23.511848   
EMERALDS     208                208       0.020957        47.458937   

          median_wait_ticks  std_wait_ticks  total_quantity  mean_quantity  \
symbol                                                                       
TOMATOES               16.0       23.374847          1442.0       3.408983   
EMERALDS               33.0       47.455211          1149.0       5.524038   

           price_mean  price_std  
symbol                            
TOMATOES  4977.716312  15.937820  
EMERALDS  9999.923077   7.863193  

Additional timing signal:
- TOMATOES: p_tick=0.0426, mean wait=23.51, CV(wait)=0.994, multi-trade timestamps=0
- EMERALDS: p_tick=0.0210, mean w

In [15]:
import numpy as np
import pandas as pd
from pathlib import Path
import re


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = (z / denom) * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))
    return center - half, center + half


def day_from_name(name):
    m = re.search(r"day_(-?\d+)", name)
    return int(m.group(1)) if m else np.nan


symbols = ["TOMATOES", "EMERALDS"]
files = sorted(Path(".").glob("trades_round_0_day_*.csv"))
tick_size = 100

rows = []
for f in files:
    df = pd.read_csv(f, sep=";")
    df["timestamp"] = df["timestamp"].astype(int)

    tmin, tmax = int(df["timestamp"].min()), int(df["timestamp"].max())
    total_ticks = len(np.arange(tmin, tmax + tick_size, tick_size))

    for sym in symbols:
        sdf = df[df["symbol"] == sym]
        active_ticks = int(sdf["timestamp"].nunique()) if not sdf.empty else 0
        p = active_ticks / total_ticks if total_ticks > 0 else np.nan
        ci_low, ci_high = wilson_ci(active_ticks, total_ticks)

        rows.append(
            {
                "file": f.name,
                "day": day_from_name(f.name),
                "symbol": sym,
                "active_ticks": active_ticks,
                "total_ticks": total_ticks,
                "p": p,
                "ci_low": ci_low,
                "ci_high": ci_high,
            }
        )

out = pd.DataFrame(rows).sort_values(["day", "symbol"]).reset_index(drop=True)

print("P values by file (trade-occurrence probability per 100-tick)")
print("p = P(at least one trade for symbol in a tick)")
print()

for file_name, g in out.groupby("file", sort=False):
    day = g["day"].iloc[0]
    print(f"File: {file_name} (day {int(day)})")
    show = g[["symbol", "active_ticks", "total_ticks", "p", "ci_low", "ci_high"]].copy()
    show["p_pct"] = 100 * show["p"]
    show["ci_pct"] = show.apply(lambda r: f"[{100*r['ci_low']:.2f}%, {100*r['ci_high']:.2f}%]", axis=1)
    print(show[["symbol", "active_ticks", "total_ticks", "p", "p_pct", "ci_pct"]].to_string(index=False))

    t = g[g["symbol"] == "TOMATOES"].iloc[0]
    e = g[g["symbol"] == "EMERALDS"].iloc[0]
    diff = t["p"] - e["p"]
    ratio = t["p"] / e["p"] if e["p"] > 0 else np.nan
    print(f"  difference (TOMATOES - EMERALDS): {diff:.4f} ({100*diff:.2f} percentage points)")
    print(f"  ratio (TOMATOES / EMERALDS): {ratio:.3f}")
    print("-" * 80)

print("\nQuick pivot (p):")
print(out.pivot(index="day", columns="symbol", values="p").sort_index())

P values by file (trade-occurrence probability per 100-tick)
p = P(at least one trade for symbol in a tick)

File: trades_round_0_day_-2.csv (day -2)
  symbol  active_ticks  total_ticks        p    p_pct         ci_pct
EMERALDS           191         9909 0.019275 1.927541 [1.67%, 2.22%]
TOMATOES           396         9909 0.039964 3.996367 [3.63%, 4.40%]
  difference (TOMATOES - EMERALDS): 0.0207 (2.07 percentage points)
  ratio (TOMATOES / EMERALDS): 2.073
--------------------------------------------------------------------------------
File: trades_round_0_day_-1.csv (day -1)
  symbol  active_ticks  total_ticks        p    p_pct         ci_pct
EMERALDS           208         9925 0.020957 2.095718 [1.83%, 2.40%]
TOMATOES           423         9925 0.042620 4.261965 [3.88%, 4.68%]
  difference (TOMATOES - EMERALDS): 0.0217 (2.17 percentage points)
  ratio (TOMATOES / EMERALDS): 2.034
--------------------------------------------------------------------------------

Quick pivot (p):
symbo

Looks like p(trade) is 0.02 for emeralds and 0.04 for tomatoes on any given day

In [16]:
import numpy as np
import pandas as pd
from pathlib import Path
import re


def day_from_name(name):
    m = re.search(r"day_(-?\d+)", name)
    return int(m.group(1)) if m else np.nan


def classify_vs_mid(trades_file, prices_file):
    tr = pd.read_csv(trades_file, sep=";")
    pr = pd.read_csv(prices_file, sep=";")

    tr["timestamp"] = tr["timestamp"].astype(int)
    pr["timestamp"] = pr["timestamp"].astype(int)

    # Keep only required columns from prices and align key names.
    pr = pr[["timestamp", "product", "mid_price"]].rename(columns={"product": "symbol"})

    merged = tr.merge(pr, on=["timestamp", "symbol"], how="left")
    merged["mid_match_found"] = merged["mid_price"].notna()

    # Label each trade relative to same-tick mid-price.
    merged["vs_mid"] = np.select(
        [merged["price"] > merged["mid_price"], merged["price"] < merged["mid_price"]],
        ["above", "below"],
        default="equal_or_missing",
    )

    # Separate exact equal from missing for transparency.
    merged.loc[merged["mid_match_found"] & (merged["price"] == merged["mid_price"]), "vs_mid"] = "equal"
    merged.loc[~merged["mid_match_found"], "vs_mid"] = "missing_mid"

    return merged


trade_files = sorted(Path(".").glob("trades_round_0_day_*.csv"))
rows = []

for tf in trade_files:
    day = day_from_name(tf.name)
    pf = Path(f"prices_round_0_day_{day}.csv")

    if not pf.exists():
        print(f"Skipping {tf.name}: no matching prices file {pf.name}")
        continue

    m = classify_vs_mid(str(tf), str(pf))

    # File-level symbol breakdown of trade counts and percentages.
    g = (
        m.groupby(["symbol", "vs_mid"], as_index=False)
        .size()
        .rename(columns={"size": "trades"})
    )
    total_by_symbol = g.groupby("symbol", as_index=False)["trades"].sum().rename(columns={"trades": "symbol_total"})
    g = g.merge(total_by_symbol, on="symbol", how="left")
    g["pct_of_symbol_trades"] = 100 * g["trades"] / g["symbol_total"]

    g["file"] = tf.name
    g["day"] = day

    # Optional: quantity-weighted perspective
    q = (
        m.groupby(["symbol", "vs_mid"], as_index=False)["quantity"]
        .sum()
        .rename(columns={"quantity": "qty"})
    )
    q_tot = q.groupby("symbol", as_index=False)["qty"].sum().rename(columns={"qty": "symbol_qty_total"})
    q = q.merge(q_tot, on="symbol", how="left")
    q["pct_of_symbol_qty"] = 100 * q["qty"] / q["symbol_qty_total"]

    out = g.merge(q[["symbol", "vs_mid", "qty", "pct_of_symbol_qty"]], on=["symbol", "vs_mid"], how="left")
    rows.append(out)

if not rows:
    print("No matching trade/price file pairs found.")
else:
    result = pd.concat(rows, ignore_index=True)
    result = result.sort_values(["day", "file", "symbol", "vs_mid"]).reset_index(drop=True)

    print("Trade price vs same-tick mid-price (by file)")
    print("Categories: above, below, equal, missing_mid")
    print()

    for file_name, gf in result.groupby("file", sort=False):
        day = int(gf["day"].iloc[0])
        print(f"File: {file_name} (day {day})")

        for sym, gs in gf.groupby("symbol", sort=False):
            print(f"  Symbol: {sym}")
            show = gs[["vs_mid", "trades", "pct_of_symbol_trades", "qty", "pct_of_symbol_qty"]].copy()
            show["pct_of_symbol_trades"] = show["pct_of_symbol_trades"].map(lambda x: f"{x:.2f}%")
            show["pct_of_symbol_qty"] = show["pct_of_symbol_qty"].map(lambda x: f"{x:.2f}%")
            print(show.to_string(index=False))
            print()

        print("-" * 84)

    print("\nCompact table:")
    print(result[["file", "day", "symbol", "vs_mid", "trades", "pct_of_symbol_trades", "qty", "pct_of_symbol_qty"]])

Trade price vs same-tick mid-price (by file)
Categories: above, below, equal, missing_mid

File: trades_round_0_day_-2.csv (day -2)
  Symbol: EMERALDS
vs_mid  trades pct_of_symbol_trades  qty pct_of_symbol_qty
 above      91               47.64%  499            47.98%
 below     100               52.36%  541            52.02%

  Symbol: TOMATOES
vs_mid  trades pct_of_symbol_trades  qty pct_of_symbol_qty
 above     189               47.61%  693            49.11%
 below     208               52.39%  718            50.89%

------------------------------------------------------------------------------------
File: trades_round_0_day_-1.csv (day -1)
  Symbol: EMERALDS
vs_mid  trades pct_of_symbol_trades  qty pct_of_symbol_qty
 above     104               50.00%  578            50.30%
 below     104               50.00%  571            49.70%

  Symbol: TOMATOES
vs_mid  trades pct_of_symbol_trades  qty pct_of_symbol_qty
 above     198               46.81%  676            46.88%
 below     225

In [17]:
import numpy as np
import pandas as pd
from pathlib import Path
import math

# Test whether trades are 50/50 above vs below mid-price.
# We ignore equal and missing_mid rows for this hypothesis.


def two_sided_binom_pvalue(x, n, p0=0.5):
    """Exact two-sided p-value using binomial pmf ordering."""
    if n <= 0:
        return np.nan

    def pmf(k):
        return math.comb(n, k) * (p0 ** k) * ((1 - p0) ** (n - k))

    px = pmf(x)
    p = 0.0
    for k in range(n + 1):
        pk = pmf(k)
        if pk <= px + 1e-15:
            p += pk
    return min(1.0, p)


rows = []
for tf in sorted(Path(".").glob("trades_round_0_day_*.csv")):
    day = day_from_name(tf.name)
    pf = Path(f"prices_round_0_day_{day}.csv")
    if not pf.exists():
        continue

    m = classify_vs_mid(str(tf), str(pf))

    for sym in ["TOMATOES", "EMERALDS"]:
        s = m[(m["symbol"] == sym) & (m["vs_mid"].isin(["above", "below"]))].copy()
        n = len(s)
        x = int((s["vs_mid"] == "above").sum())

        if n < 2:
            rows.append(
                {
                    "file": tf.name,
                    "day": day,
                    "symbol": sym,
                    "n_above_below": n,
                    "n_above": x,
                    "p_above": np.nan,
                    "t_stat": np.nan,
                    "p_t_two_sided": np.nan,
                    "p_binom_two_sided": np.nan,
                    "decision_5pct": "insufficient_data",
                }
            )
            continue

        y = (s["vs_mid"] == "above").astype(float).to_numpy()
        p_hat = y.mean()
        s_std = y.std(ddof=1)

        # One-sample t-test vs mean 0.5
        if s_std == 0:
            t_stat = np.inf if p_hat > 0.5 else (-np.inf if p_hat < 0.5 else 0.0)
            p_t = 0.0 if p_hat != 0.5 else 1.0
        else:
            t_stat = (p_hat - 0.5) / (s_std / np.sqrt(n))
            try:
                from scipy import stats as st
                p_t = 2 * st.t.sf(abs(t_stat), df=n - 1)
            except Exception:
                # Normal approximation fallback if scipy is unavailable.
                z = abs(t_stat)
                p_t = math.erfc(z / np.sqrt(2))

        p_binom = two_sided_binom_pvalue(x, n, p0=0.5)

        decision = "reject_50_50" if p_t < 0.05 else "cannot_reject_50_50"

        rows.append(
            {
                "file": tf.name,
                "day": day,
                "symbol": sym,
                "n_above_below": n,
                "n_above": x,
                "p_above": p_hat,
                "t_stat": t_stat,
                "p_t_two_sided": p_t,
                "p_binom_two_sided": p_binom,
                "decision_5pct": decision,
            }
        )

res = pd.DataFrame(rows).sort_values(["day", "symbol"]).reset_index(drop=True)

print("50/50 test: above vs below mid-price")
print("H0: p_above = 0.5  (two-sided)")
print("Uses: one-sample t-test on indicator + exact binomial check")
print()

print(res[[
    "file", "day", "symbol", "n_above_below", "n_above", "p_above",
    "t_stat", "p_t_two_sided", "p_binom_two_sided", "decision_5pct"
]])

print("\nBy file summary:")
for file_name, g in res.groupby("file", sort=False):
    day = int(g["day"].iloc[0])
    print(f"File: {file_name} (day {day})")
    for _, r in g.iterrows():
        print(
            f"  {r['symbol']}: p_above={r['p_above']:.3f}, n={int(r['n_above_below'])}, "
            f"t_p={r['p_t_two_sided']:.4g}, binom_p={r['p_binom_two_sided']:.4g}, "
            f"{r['decision_5pct']}"
        )
    print("-" * 84)

50/50 test: above vs below mid-price
H0: p_above = 0.5  (two-sided)
Uses: one-sample t-test on indicator + exact binomial check

                        file  day    symbol  n_above_below  n_above   p_above  \
0  trades_round_0_day_-2.csv   -2  EMERALDS            191       91  0.476440   
1  trades_round_0_day_-2.csv   -2  TOMATOES            397      189  0.476071   
2  trades_round_0_day_-1.csv   -1  EMERALDS            208      104  0.500000   
3  trades_round_0_day_-1.csv   -1  TOMATOES            423      198  0.468085   

     t_stat  p_t_two_sided  p_binom_two_sided        decision_5pct  
0 -0.650232       0.516328           0.562798  cannot_reject_50_50  
1 -0.953474       0.340932           0.366336  cannot_reject_50_50  
2  0.000000       1.000000           1.000000  cannot_reject_50_50  
3 -1.313912       0.189590           0.206118  cannot_reject_50_50  

By file summary:
File: trades_round_0_day_-2.csv (day -2)
  EMERALDS: p_above=0.476, n=191, t_p=0.5163, binom_p=0.5628,